# Explore CIFAR-10 image embeddings

Load the saved high-dimensional image vectors produced by [Prepare CIFAR-10 EmbeddingGemma 2 vectors](../data/cifar10/prepare_dataset.ipynb), then compare recursive UMAP with IMSTE’s multilevel start and integrated gravity result. The 2D UMAP and IMSTE layouts are recomputed on every run.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "imste" / "__init__.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not locate this IMSTE repository from the notebook working directory.")
sys.path.insert(0, str(REPO_ROOT))
if getattr(sys.modules.get("imste"), "__file__", None) is None:
    sys.modules.pop("imste", None)
DATA_DIR = REPO_ROOT / "data" / "cifar10"
print("Repository:", REPO_ROOT)
print("CIFAR-10 data:", DATA_DIR)


Repository: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree
CIFAR-10 data: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree/data/cifar10


In [2]:
import hashlib
import json
import time

import numpy as np
import pandas as pd

MANIFEST_PATH = DATA_DIR / "cache" / "embeddinggemma2_cifar10_manifest.json"
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {MANIFEST_PATH}. Run ../data/cifar10/prepare_dataset.ipynb first."
    )
manifest = json.loads(MANIFEST_PATH.read_text())
EMBEDDING_PATH = MANIFEST_PATH.parent / manifest["embedding_file"]
with np.load(EMBEDDING_PATH, allow_pickle=False) as saved:
    X = saved["embeddings"]
    labels = saved["labels"]
    sample_indices = saved["indices"]
class_names = manifest["class_names"]

if X.ndim != 2 or X.shape[1] != 768 or len(X) != len(labels):
    raise ValueError(f"Unexpected vector/label shapes: {X.shape}, {labels.shape}")
if not np.isfinite(X).all():
    raise ValueError("The saved image vectors contain NaN or infinite values.")
actual_fingerprint = hashlib.sha256(np.ascontiguousarray(X).view(np.uint8)).hexdigest()
if actual_fingerprint != manifest["vector_sha256"]:
    raise ValueError("The saved vectors do not match the manifest fingerprint.")

print("Dataset:", manifest["dataset"])
print("Encoder:", manifest["model_id"])
print("Vector matrix:", X.shape, X.dtype)
print("Loaded from:", EMBEDDING_PATH)


Dataset: CIFAR-10 train
Encoder: google/embeddinggemma-2
Vector matrix: (50000, 768) float32
Loaded from: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/IteratedMinimumSpanningTree/data/cifar10/cache/cifar10_train_5000_per_class_seed42_embeddinggemma_2.npz


## 1. Fit recursive UMAP and IMSTE

Both layouts use the same normalized image vectors. CIFAR-10 labels are reserved for coloring the final plots.

In [3]:
import hashlib
import json
import time



In [4]:
import umap

UMAP_SETTINGS = {
    "n_components": 2,
    "n_neighbors": 15,
    "min_dist": 0.1,
    "metric": "euclidean",
    "init": "recursive",
    "recursive_coarsening_ratio": 4,
    "compatibility_layout": False,
    "nearest_neighbor_backend": "rinnd",
    "random_state": int(manifest["random_state"]),
    "n_epochs": 200,
    "repulsion_strength": 1.0,
    "verbose": True,
}

umap_started = time.perf_counter()
Z_umap = umap.UMAP(**UMAP_SETTINGS).fit_transform(X)
umap_seconds = time.perf_counter() - umap_started
print(f"UMAP fit in {umap_seconds / 60:.1f} min")
print("UMAP coordinates:", Z_umap.shape)


UMAP(compatibility_layout=False, init='recursive', n_epochs=200, nearest_neighbor_backend='rinnd', random_state=42, verbose=True)
Sat Nov 14 18:38:54 2026 Construct fuzzy simplicial set
Sat Nov 14 18:38:54 2026 Finding Nearest Neighbors
Sat Nov 14 18:38:54 2026 Using RINND to find nearest neighbors
Building RP forest with 8 trees...
Initializing graph from 9692 leaves...
Running NN-descent for up to 16 iterations...
Iteration 1: 278174 updates
Iteration 2: 113301 updates
Iteration 3: 20867 updates
Iteration 4: 3936 updates
Iteration 5: 843 updates
Iteration 6: 202 updates
Converged after 6 iterations

=== Timing Breakdown ===
Forest building:     121.829ms
Leaf initialization: 115.695ms
Candidate building:   25.009ms (6 iters)
Update iterations:   288.001ms (6 iters)
Total measured:      550.533ms
Sat Nov 14 18:38:54 2026 Finished Nearest Neighbor Search
Sat Nov 14 18:38:56 2026 Construct embedding
Sat Nov 14 18:38:56 2026 Recursive initialization: coarsening 50000 nodes into about 125

/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users

Init recursion depth 3:   0%|          | 0/98 [00:00<?, ?it/s]

Init recursion depth 2:   0%|          | 0/70 [00:00<?, ?it/s]

/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/umap/label_prop.py:959: RuntimeWarning: divide by zero encountered in matmul
  result = result @ rotation
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/umap/label_prop.py:959: RuntimeWarning: overflow encountered in matmul
  result = result @ rotation
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/umap/label_prop.py:959: RuntimeWarning: invalid value encountered in matmul
  result = result @ rotation


Init recursion depth 1:   0%|          | 0/50 [00:00<?, ?it/s]

Sat Nov 14 18:38:57 2026 Using adam layout optimizer


Epochs completed:   0%|            0/200 [00:00]

Sat Nov 14 18:38:58 2026 Finished embedding
UMAP fit in 0.1 min
UMAP coordinates: (50000, 2)


In [ ]:
from imste import IteratedMinimumSpanningTreeEmbedder

IMSTE_N_MSTS = 3
IMSTE_N_EPOCHS = 200
IMSTE_GRAVITY_STRENGTH = 0.2
IMSTE_MEAN_SHIFT_ITERATIONS = 1
IMSTE_SETTINGS = {
    "n_msts": IMSTE_N_MSTS,
    "n_epochs": IMSTE_N_EPOCHS,
    "random_state": int(manifest["random_state"]),
    "gravity_strength": IMSTE_GRAVITY_STRENGTH,
    "mean_shift_iterations": IMSTE_MEAN_SHIFT_ITERATIONS,
}

imste_estimator = IteratedMinimumSpanningTreeEmbedder(**IMSTE_SETTINGS)
Z_imste_base, Z_imste = imste_estimator.fit_transform(X, return_intermediate=True)
imste_seconds = imste_estimator.imste_fit_time_
gravity_seconds = imste_estimator.gravity_time_
mean_shift_seconds = imste_estimator.mean_shift_time_
print(f"IMSTE fit {imste_seconds:.1f}s; gravity {gravity_seconds:.1f}s; mean shift {mean_shift_seconds:.1f}s")


## 2. View the embeddings

Each point is one image. Color indicates its CIFAR-10 class.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from imste.visualization import render_datashader_embedding
labels_for_plot = np.asarray([class_names[int(label)] for label in labels])
palette = plt.get_cmap("tab10")
color_key = {name: to_hex(palette(i)) for i, name in enumerate(class_names)}
layouts = [
    ("UMAP · recursive init", Z_umap, umap_seconds),
    ("IMSTE · multilevel start", Z_imste_base, imste_seconds),
    ("IMSTE · gravity", Z_imste, imste_seconds + gravity_seconds),
]
fig, axes = plt.subplots(1, len(layouts), figsize=(8 * len(layouts), 7), constrained_layout=True)
for ax, (name, coordinates, elapsed) in zip(axes, layouts):
    image = render_datashader_embedding(
        coordinates,
        labels_for_plot,
        width=1400,
        height=1200,
        color_key=color_key,
        min_alpha=40,
        background="white",
        spread_px=2,
    )
    ax.imshow(image.to_pil())
    runtime = f"{elapsed / 60:.1f} min" if elapsed >= 60 else f"{elapsed:.1f} s"
    ax.set_title(f"CIFAR-10 — {name} ({runtime})")
    ax.set_axis_off()
plt.show()
